# E0 — IPAD R01 Frame Baseline (Google Colab)

Bu notebook frozen ImageNet ResNet18 ile frame embedding çıkarır. Train/validation ayrımı sequence bazındadır; threshold yalnızca normal validation skorlarının 0.995 quantile değeridir. Resmî test verisi threshold belirlemede kullanılmaz. Full eğitim ancak tüm hücreleri bilinçli olarak çalıştırdığınızda başlar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Tek noktadan düzenlenecek yollar ve çalışma ayarları
PROJECT_ROOT = '/content/drive/MyDrive/bitirme'
DATASET_ROOT = '/content/drive/MyDrive/bitirme_dataset/R01'
REBUILD_EMBEDDINGS = False
BATCH_SIZE = 64                 # CUDA OOM olursa 32, ardından 16
NUM_WORKERS = 2
SEED = 42
IMAGE_SIZE = 256
KNN_K = 5
THRESHOLD_QUANTILE = 0.995
MAX_MEMORY_SAMPLES = None       # None: tüm normal train embeddingleri

In [ ]:
import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GiB')
else:
    print('GPU unavailable; this notebook will use CPU (slow).')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('selected device:', DEVICE)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT_ROOT}/requirements.txt'], check=True)

In [ ]:
import sys, os, json, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
from src.data.ipad_r01_dataset import R01FrameDataset
from src.models.frame_baseline import ResNet18FeatureExtractor, knn_scores
from src.training.train_frame_baseline import validate_manifest
from src.evaluation.evaluate_frame import exact_test_alignment
from src.evaluation.metrics import calculate_metrics
from src.utils.seed import set_seed
from torchvision import transforms
from torch.utils.data import DataLoader
print('Project imports OK:', PROJECT_ROOT)

In [ ]:
# Dataset integrity and manifest; regenerate stale local-Mac paths after Drive upload
root = Path(DATASET_ROOT)
assert (root/'training/frames').is_dir(), f'Missing training/frames under {root}'
assert (root/'testing/frames').is_dir(), f'Missing testing/frames under {root}'
assert (root/'test_label').is_dir(), f'Missing test_label under {root}'
manifest_path = Path(PROJECT_ROOT)/'data/processed/r01_manifest.csv'
regenerate = not manifest_path.exists()
if not regenerate:
    old = pd.read_csv(manifest_path)
    regenerate = not old.path.map(lambda p: Path(p).is_file()).all()
if regenerate:
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, 'scripts/prepare_r01.py', '--root', DATASET_ROOT, '--output', str(manifest_path)], cwd=PROJECT_ROOT, check=True)
manifest = pd.read_csv(manifest_path)
train_df, val_df, _ = validate_manifest(manifest)
test_df = manifest[manifest.split == 'test'].copy()
assert len(train_df)==6196 and train_df.sequence.nunique()==27, (len(train_df),train_df.sequence.nunique())
assert len(val_df)==1612 and val_df.sequence.nunique()==7, (len(val_df),val_df.sequence.nunique())
assert len(test_df)==3685 and test_df.sequence.nunique()==15, (len(test_df),test_df.sequence.nunique())
assert (train_df.label==0).all() and (val_df.label==0).all()
assert not (set(train_df.sequence)&set(val_df.sequence))
print('Train:', train_df.sequence.nunique(), 'sequences /', len(train_df), 'normal frames')
print('Validation:', val_df.sequence.nunique(), 'sequences /', len(val_df), 'normal frames')
print('Test:', test_df.sequence.nunique(), 'sequences /', len(test_df), 'frames')
display(manifest.groupby(['split','label'],dropna=False).size().rename('frames').to_frame())

In [ ]:
# Verify official .npy label mapping and frame counts directly from Drive
for seq_id in range(1,16):
    seq=f'{seq_id:02d}'; label_file=root/'test_label'/f'{seq_id:03d}.npy'; frame_dir=root/'testing'/'frames'/seq
    assert label_file.is_file() and frame_dir.is_dir(), f'Missing label/frames for {seq}'
    labels=np.load(label_file, allow_pickle=False).reshape(-1)
    frame_files=sorted(frame_dir.glob('*.jpg'), key=lambda p:int(p.stem))
    rows=test_df[test_df.sequence==f'testing/{seq}']
    assert len(labels)==len(frame_files)==len(rows), f'{seq}: labels={len(labels)}, frames={len(frame_files)}, manifest={len(rows)}'
    lookup=dict(zip([p.name for p in frame_files],labels.astype(int)))
    for r in rows.itertuples(): assert int(r.label)==lookup[r.frame_name], f'Label alignment error {seq}/{r.frame_name}'
print('All 15 test label files match their frame folders and manifest exactly.')

In [ ]:
# Create reproducible Drive run and record exact settings
from datetime import datetime, timezone
set_seed(SEED)
timestamp=datetime.now(timezone.utc).astimezone().strftime('%Y%m%d_%H%M%S')
RUN_DIR=Path(PROJECT_ROOT)/'models'/f'E0_frame_resnet18_{timestamp}'
EMBEDDING_CACHE_DIR=Path(PROJECT_ROOT)/'outputs'/'embedding_cache'/'E0_R01_resnet18'
EMBEDDING_CACHE_DIR.mkdir(parents=True,exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR/'figures').mkdir()
print('Run directory:', RUN_DIR)

In [ ]:
# Load frozen pretrained ResNet18 (512-D global pooled feature)
model=ResNet18FeatureExtractor(pretrained=True).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad=False
assert model.embedding_dim==512
transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
print('Embedding dimension:',model.embedding_dim,'; input:',IMAGE_SIZE,'x',IMAGE_SIZE,'; device:',DEVICE)

In [ ]:
# Resumable embedding extraction. Cache keys are checked before reuse.
def extract_embeddings(records, cache_name):
    cache_path=EMBEDDING_CACHE_DIR/cache_name
    run_copy=RUN_DIR/cache_name
    expected_keys=records[['sequence','frame_name']].astype(str).to_dict('records')
    signature={'image_size':IMAGE_SIZE,'embedding_dim':512,'normalized':True,'weights':'ResNet18_Weights.DEFAULT'}
    if cache_path.exists() and not REBUILD_EMBEDDINGS:
        cached=torch.load(cache_path,map_location='cpu',weights_only=True)
        if cached.get('keys')==expected_keys and cached.get('signature')==signature:
            import shutil
            shutil.copy2(cache_path,run_copy)
            print('Using cached embeddings:',cache_path)
            return cached['embeddings'].float()
        print('Cache keys differ; recomputing',cache_path)
    batch=BATCH_SIZE
    while batch>=16:
        try:
            ds=R01FrameDataset(records,transform)
            loader=DataLoader(ds,batch_size=batch,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),persistent_workers=(NUM_WORKERS>0))
            parts=[]
            with torch.inference_mode():
                for images,_ in loader:
                    z=model(images.to(DEVICE,non_blocking=(DEVICE.type=='cuda'))).float()
                    z=torch.nn.functional.normalize(z,p=2,dim=1)
                    parts.append(z.cpu())
            embeddings=torch.cat(parts).to(torch.float32)
            torch.save({'keys':expected_keys,'signature':signature,'embeddings':embeddings},cache_path)
            import shutil
            shutil.copy2(cache_path,run_copy)
            return embeddings
        except torch.cuda.OutOfMemoryError:
            if DEVICE.type!='cuda' or batch==16: raise
            batch//=2; torch.cuda.empty_cache(); print('CUDA OOM; retrying with batch_size=',batch)
    raise RuntimeError('Embedding extraction failed')
train_embeddings=extract_embeddings(train_df,'train_embeddings.pt')
validation_embeddings=extract_embeddings(val_df,'validation_embeddings.pt')
assert train_embeddings.shape==(len(train_df),512) and train_embeddings.dtype==torch.float32
assert validation_embeddings.shape==(len(val_df),512)
print('Train memory candidate:',tuple(train_embeddings.shape),'Validation:',tuple(validation_embeddings.shape))

In [ ]:
# Normal-only memory bank and validation threshold; no test frames/labels used here
memory_bank=train_embeddings
if MAX_MEMORY_SAMPLES is not None and len(memory_bank)>MAX_MEMORY_SAMPLES:
    g=torch.Generator().manual_seed(SEED); memory_bank=memory_bank[torch.randperm(len(memory_bank),generator=g)[:MAX_MEMORY_SAMPLES]]
validation_scores=knn_scores(validation_embeddings,memory_bank,k=KNN_K).numpy().astype(float)
threshold=float(np.quantile(validation_scores,THRESHOLD_QUANTILE))
threshold_info={'threshold':threshold,'method':'quantile','quantile':THRESHOLD_QUANTILE,'num_validation_frames':int(len(validation_scores)),'validation_score_mean':float(validation_scores.mean()),'validation_score_std':float(validation_scores.std()),'validation_score_min':float(validation_scores.min()),'validation_score_max':float(validation_scores.max()),'validation_score_p95':float(np.quantile(validation_scores,.95)),'validation_score_p99':float(np.quantile(validation_scores,.99)),'validation_score_p995':float(np.quantile(validation_scores,.995)),'test_used_for_threshold':False}
(RUN_DIR/'threshold.json').write_text(json.dumps(threshold_info,indent=2))
torch.save(memory_bank.to(torch.float32),RUN_DIR/'memory_bank.pt')
print(json.dumps(threshold_info,indent=2))

In [ ]:
# Save run configuration and provenance
config={'seed':SEED,'dataset_root':DATASET_ROOT,'project_root':PROJECT_ROOT,'train_sequences':sorted(train_df.sequence.unique().tolist()),'validation_sequences':sorted(val_df.sequence.unique().tolist()),'test_sequences':sorted(test_df.sequence.unique().tolist()),'num_train_frames':len(train_df),'num_validation_frames':len(val_df),'num_test_frames':len(test_df),'backbone':'resnet18','weights':'ResNet18_Weights.DEFAULT','embedding_dim':512,'image_size':IMAGE_SIZE,'normalize_embeddings':True,'score_method':'knn_euclidean_mean','knn_k':KNN_K,'threshold_method':'quantile','threshold_quantile':THRESHOLD_QUANTILE,'batch_size':BATCH_SIZE,'num_workers':NUM_WORKERS,'device':str(DEVICE),'timestamp':timestamp,'rebuild_embeddings':REBUILD_EMBEDDINGS}
(RUN_DIR/'config.json').write_text(json.dumps(config,indent=2))
print('Saved config.json')

In [ ]:
# Test inference starts only after threshold.json is fixed
test_embeddings=extract_embeddings(test_df,'test_embeddings.pt')
test_scores=knn_scores(test_embeddings,memory_bank,k=KNN_K).numpy().astype(float)
predictions=test_df[['split','sequence','frame_name','frame_index']].reset_index(drop=True).copy()
predictions['ground_truth']=test_df.label.astype(int).to_numpy()
predictions['anomaly_score']=test_scores
predictions['threshold']=threshold
predictions['prediction']=(test_scores>=threshold).astype(int)
aligned=exact_test_alignment(predictions,test_df.rename(columns={'label':'ground_truth'}))
aligned=aligned[['sequence','frame_name','frame_index','ground_truth','anomaly_score','threshold','prediction']]
aligned.to_csv(RUN_DIR/'predictions.csv',index=False)
print('Exact test alignment OK:',len(aligned),'predictions')

In [ ]:
# Final test metrics (threshold remains unchanged)
metrics=calculate_metrics(aligned.ground_truth,aligned.anomaly_score,threshold)
metrics.update({'threshold':threshold,'threshold_method':'quantile','threshold_quantile':THRESHOLD_QUANTILE,'num_test_frames':len(aligned),'test_used_for_threshold':False})
(RUN_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2))
display(pd.DataFrame([metrics]).T.rename(columns={0:'value'}))

In [ ]:
# Confusion matrix and score distribution figures
from sklearn.metrics import confusion_matrix
tn,fp,fn,tp=metrics['tn'],metrics['fp'],metrics['fn'],metrics['tp']
fig,ax=plt.subplots(figsize=(5,4)); mat=np.array([[tn,fp],[fn,tp]]); im=ax.imshow(mat,cmap='Blues')
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Normal','Anomaly'],yticklabels=['Normal','Anomaly'],xlabel='Predicted',ylabel='Ground truth',title='E0 Confusion Matrix')
for (i,j),v in np.ndenumerate(mat): ax.text(j,i,str(v),ha='center',va='center')
fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(RUN_DIR/'figures'/'confusion_matrix.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(8,4))
for label,name,color in [(0,'Normal test','tab:blue'),(1,'Anomaly test','tab:red')]:
    values=aligned.loc[aligned.ground_truth==label,'anomaly_score']; ax.hist(values,bins=40,alpha=.55,density=True,label=name,color=color)
ax.axvline(threshold,color='black',linestyle='--',label=f'Validation threshold={threshold:.4g}')
ax.set(xlabel='Anomaly score',ylabel='Density',title='E0 Test Score Distribution'); ax.legend(); fig.tight_layout(); fig.savefig(RUN_DIR/'figures'/'anomaly_score_distribution.png',dpi=160); plt.show()

In [ ]:
# Human-readable run summary and output location
summary=f'''# E0 R01 Frame Baseline

- Timestamp: {timestamp}
- Dataset: `{DATASET_ROOT}`
- Device: {DEVICE}
- Backbone: frozen ImageNet pretrained ResNet18, 512-D pooled features
- Train: {train_df.sequence.nunique()} normal sequences / {len(train_df)} frames
- Validation: {val_df.sequence.nunique()} held-out normal sequences / {len(val_df)} frames
- Test: {test_df.sequence.nunique()} official sequences / {len(test_df)} labeled frames
- Scoring: L2-normalized embeddings, Euclidean k-NN mean distance, k={KNN_K}
- Threshold: validation normal {THRESHOLD_QUANTILE} quantile = {threshold:.8f}
- Test labels were used only for final metrics.

## Metrics

```json
{json.dumps(metrics,indent=2)}
```
'''
(RUN_DIR/'run_summary.md').write_text(summary)
print('Completed E0 run:',RUN_DIR)
print('Files:',*[p.name for p in sorted(RUN_DIR.iterdir())],sep='\n- ')